# MobileNetV2 — Inverted Residuals and Linear Bottlenecks (2018)

_Papers · Computer Vision_

**Expand thin channels, filter cheaply, project back down with NO ReLU, and skip-connect the thin ends.**

---

This notebook builds MobileNetV2's inverted-residual block from scratch, one piece at a time. Run each cell, read the note above it, and watch why leaving the thin bottleneck *linear* — no ReLU — lets a tiny net learn far more than the same net with a ReLU there. Then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Count the cost of one inverted-residual block

We start with plain arithmetic on the lesson's worked example: a `14x14` map, thin in/out widths `d'=d''=24`, expansion factor `t=6`, kernel `k=3`. The block does three things in sequence — expand the channels with a `1x1` (cost `h·w·d'·t·d'`), filter the wide tensor depthwise (`h·w·(t·d')·k²`), and project back down with a `1x1` (`h·w·(t·d')·d''`). Adding those gives a total that matches the closed form `h·w·d'·t·(d'+k²+d'')`, which we verify here.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

# Recompute the worked example: 14x14 map, d'=d''=24, t=6, k=3.
h = w = 14
dp = 24
dpp = 24
t = 6
k = 3

wide = t * dp                    # wide middle channels = 144
expand = h * w * dp * wide       # 1x1  d'->t*d'
dwise = h * w * wide * k * k     # 3x3 depthwise on wide
proj = h * w * wide * dpp        # 1x1  t*d'->d''
block = expand + dwise + proj
formula = h * w * dp * t * (dp + k * k + dpp)

print("wide channels =", wide)
print("expand =", expand, " depthwise =", dwise, " project =", proj, " block =", block)
print("closed form h*w*d'*t*(d'+k^2+d'') =", formula, " (match:", block == formula, ")")
# wide channels = 144
# expand = 677376  depthwise = 254016  project = 677376  block = 1608768
# closed form ... = 1608768  (match: True)

### Step 2 — Build the inverted-residual + linear-bottleneck block

Here is the block as a PyTorch module. It expands with a `1x1` (ReLU6), filters depthwise (ReLU6), then projects back down — and crucially the projection has **no activation**, the "linear bottleneck." A `linear=False` flag adds a ReLU6 there instead, giving us the ablation. The skip connection (`use_res`) only fires when stride is 1 and the in/out widths match, so the residual links the two *thin* ends of the block — the "inverted" residual.

In [ ]:
# The inverted-residual + linear-bottleneck block (built by hand).
class InvertedResidual(nn.Module):
    # linear=True  -> bottleneck projection has NO activation (the paper's design)
    # linear=False -> ablation: ReLU6 on the thin bottleneck
    def __init__(self, in_ch, out_ch, t=6, stride=1, linear=True):
        super().__init__()
        hid = in_ch * t
        self.use_res = (stride == 1 and in_ch == out_ch)   # skip only when shapes match
        self.expand = nn.Conv2d(in_ch, hid, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(hid)
        self.dwise = nn.Conv2d(hid, hid, 3, stride=stride, padding=1, groups=hid, bias=False)
        self.bn2 = nn.BatchNorm2d(hid)
        self.proj = nn.Conv2d(hid, out_ch, 1, bias=False)
        self.bn3 = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU6(inplace=True)
        self.linear = linear

    def forward(self, x):
        h = self.act(self.bn1(self.expand(x)))     # expand 1x1  + ReLU6
        h = self.act(self.bn2(self.dwise(h)))      # depthwise 3x3 + ReLU6
        h = self.bn3(self.proj(h))                 # project 1x1 -- LINEAR bottleneck (no ReLU)
        if not self.linear:
            h = self.act(h)                        # ablation: ReLU6 on the thin bottleneck
        if self.use_res:
            h = h + x                              # inverted residual: skip between thin ends
        return h

### Step 3 — Stack blocks into a tiny net and make a harder toy task

We stack five inverted-residual blocks (thin 6–8 channel bottlenecks) on top of a small stem, ending in a global-average-pool and a linear head. Two of the blocks keep the same width at stride 1, so they get skip connections; one downsamples and widens, so it does not. We pair this with a deliberately harder 12-class toy dataset (more noise), so the thin bottleneck genuinely has to carry information — which is exactly where a ReLU there would hurt.

In [ ]:
# A tiny MobileNetV2-style net with thin bottlenecks (6 / 8 channels).
class TinyMNV2(nn.Module):
    def __init__(self, n_classes=12, linear=True):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, 6, 3, padding=1, bias=False),
                                  nn.BatchNorm2d(6), nn.ReLU6(inplace=True))
        self.b1 = InvertedResidual(6, 6, 6, 1, linear)   # stride1, same width -> has skip
        self.b2 = InvertedResidual(6, 6, 6, 1, linear)
        self.b3 = InvertedResidual(6, 8, 6, 2, linear)   # downsample + widen -> no skip
        self.b4 = InvertedResidual(8, 8, 6, 1, linear)
        self.b5 = InvertedResidual(8, 8, 6, 1, linear)
        self.head = nn.Linear(8, n_classes)

    def forward(self, x):
        x = self.b5(self.b4(self.b3(self.b2(self.b1(self.stem(x))))))
        return self.head(x.mean(dim=(2, 3)))             # global average pool -> classifier

def n_params(m):
    return sum(p.numel() for p in m.parameters())


# Harder toy task: 12 noisy classes, so the thin bottleneck must carry real info.
g = torch.Generator().manual_seed(1)
Nimg, C, H, W, K = 900, 3, 16, 16, 12
y = torch.randint(0, K, (Nimg,), generator=g)
proto = torch.randn(K, C, H, W, generator=g)
X = proto[y] + 1.3 * torch.randn(Nimg, C, H, W, generator=g)
Xtr, ytr, Xte, yte = X[:700], y[:700], X[700:], y[700:]

### Step 4 — Train the linear vs. ReLU bottleneck and compare

Now we train two copies of the same net that differ only in the bottleneck activation, and compare. Both have identical parameter counts — a ReLU adds none — so any gap is purely the activation. The linear-bottleneck net should win clearly: a ReLU on a thin layer zeroes negatives and folds away part of the data, information the net cannot recover. (Numbers vary by hardware and seed; this is our small run, not the paper's reported figure.)

In [ ]:
def train(net, epochs=140, lr=0.05):
    torch.manual_seed(0)
    opt = torch.optim.SGD(net.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4)
    lf = nn.CrossEntropyLoss()
    for _ in range(epochs):
        net.train()
        opt.zero_grad()
        lf(net(Xtr), ytr).backward()
        opt.step()
    net.eval()
    with torch.no_grad():
        return (net(Xte).argmax(1) == yte).float().mean().item()

linear_net = TinyMNV2(linear=True)    # paper's design: linear bottleneck
relu_net = TinyMNV2(linear=False)     # ablation: ReLU6 bottleneck

acc_lin = train(linear_net)
acc_rel = train(relu_net)

print("\n               params     test acc")
print("linear bottleneck  %5d     %.3f" % (n_params(linear_net), acc_lin))
print("ReLU   bottleneck  %5d     %.3f" % (n_params(relu_net), acc_rel))
print("same params (ReLU has none); linear bottleneck wins -- ReLU destroys info in the thin layer.")
# linear bottleneck ~0.77, ReLU bottleneck ~0.43 in our run (varies by hardware/seed;
# our small run, not the paper's reported number).

## Visualize the data & results

_Does a LINEAR bottleneck beat a ReLU bottleneck when the bottleneck layers are thin?_

### Step 1 — Rebuild the experiment and record accuracy curves

This visualization cell is self-contained, so it re-creates the 12-class toy data and the inverted-residual block, then trains while recording a test-accuracy *curve* sampled every few epochs. Tracking the curve (not just the final number) shows how each variant progresses over training. The block is the same as above, written compactly, with the same `linear` flag controlling whether the bottleneck gets a ReLU6.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)
g = torch.Generator().manual_seed(1)
N, C, H, W, K = 900, 3, 16, 16, 12
y = torch.randint(0, K, (N,), generator=g)
proto = torch.randn(K, C, H, W, generator=g)
X = proto[y] + 1.3 * torch.randn(N, C, H, W, generator=g)
Xtr, ytr, Xte, yte = X[:700], y[:700], X[700:], y[700:]

class IR(nn.Module):   # inverted residual; linear=False -> ReLU6 on the bottleneck (ablation)
    def __init__(s, ci, co, t=6, stride=1, linear=True):
        super().__init__()
        hid = ci * t
        s.use_res = (stride == 1 and ci == co)
        s.linear = linear
        s.e = nn.Conv2d(ci, hid, 1, bias=False)
        s.b1 = nn.BatchNorm2d(hid)
        s.d = nn.Conv2d(hid, hid, 3, stride=stride, padding=1, groups=hid, bias=False)
        s.b2 = nn.BatchNorm2d(hid)
        s.p = nn.Conv2d(hid, co, 1, bias=False)
        s.b3 = nn.BatchNorm2d(co)
        s.a = nn.ReLU6(inplace=True)
    def forward(s, x):
        h = s.a(s.b1(s.e(x)))
        h = s.a(s.b2(s.d(h)))
        h = s.b3(s.p(h))            # project = LINEAR
        if not s.linear:
            h = s.a(h)
        if s.use_res:
            h = h + x
        return h

class Net(nn.Module):
    def __init__(s, linear=True):
        super().__init__()
        s.stem = nn.Sequential(nn.Conv2d(3, 6, 3, padding=1, bias=False), nn.BatchNorm2d(6), nn.ReLU6(inplace=True))
        s.b1 = IR(6, 6, 6, 1, linear)
        s.b2 = IR(6, 6, 6, 1, linear)
        s.b3 = IR(6, 8, 6, 2, linear)
        s.b4 = IR(8, 8, 6, 1, linear)
        s.b5 = IR(8, 8, 6, 1, linear)
        s.head = nn.Linear(8, K)
    def forward(s, x):
        return s.head(s.b5(s.b4(s.b3(s.b2(s.b1(s.stem(x)))))).mean(dim=(2, 3)))

def nparams(m):
    return sum(p.numel() for p in m.parameters())

def train(net, epochs=140, lr=0.05):
    torch.manual_seed(0)
    opt = torch.optim.SGD(net.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4)
    lf = nn.CrossEntropyLoss()
    curve = []
    for e in range(epochs):
        net.train()
        opt.zero_grad()
        lf(net(Xtr), ytr).backward()
        opt.step()
        if e % 14 == 0 or e == epochs - 1:
            net.eval()
            with torch.no_grad():
                curve.append((e, round((net(Xte).argmax(1) == yte).float().mean().item(), 3)))
    return curve

### Step 2 — Train both variants and print the curves

Now we train the linear-bottleneck net and the ReLU-bottleneck net and print their accuracy curves side by side. With matched parameter counts, the linear bottleneck should climb steadily while the ReLU variant stalls low — the ReLU on the thin layer destroys information the net needs. This is the §3.3 / Fig. 6a result in miniature.

In [ ]:
lin, rel = Net(True), Net(False)
c_lin, c_rel = train(lin), train(rel)

print("params  lin=%d rel=%d (same; ReLU has none)" % (nparams(lin), nparams(rel)))
print("linear bottleneck curve:", c_lin)
print("ReLU   bottleneck curve:", c_rel)
# params lin=5910 rel=5910; linear reaches 0.765, ReLU stalls at 0.430 -- ReLU on the thin
# bottleneck destroys information. Our small run, not the paper's reported number.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation. You have a working tiny MobileNetV2 whose blocks use a linear bottleneck.
            Add a ReLU6 after every projection $1\times1$ (making the bottleneck non-linear), keep everything else
            identical, and retrain. What happens to the parameter count and the test accuracy, and what does the
            comparison demonstrate?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Change one thing only: add nn.ReLU6 after the project conv's BatchNorm. Keep depth, bottleneck widths, expansion $t$, optimizer, data, and seed identical. — _An honest ablation isolates the linear-vs-ReLU bottleneck; any accuracy gap is attributable to it alone._
- Note the parameter counts. — _A ReLU has no parameters, so the two nets have the same parameter count &mdash; the difference is purely the activation, not capacity._
- Train both and compare test accuracy. — _The linear-bottleneck net should clearly win &mdash; the paper's claim that a non-linearity in the thin bottleneck destroys information and hurts performance._

**Answer:** The two nets have identical parameter counts (a ReLU adds none), yet the linear-bottleneck
                 net reaches clearly higher test accuracy. In our run the linear bottleneck hit ~0.77 while
                 the ReLU bottleneck stalled at ~0.43 on a 12-class toy task with thin (6&ndash;8 channel)
                 bottlenecks. Because the only difference is the activation on the thin projection, this isolates
                 the linear bottleneck as the cause: a ReLU on a low-dimensional layer zeroes negatives and folds
                 away part of the data manifold, information the network cannot recover. That is exactly the
                 &sect;3.3 / Fig. 6a result. The CODEVIZ panel shows the gap.

</details>

**Problem 2.** An inverted-residual block runs on a $7\times7$ map with thin bottleneck $d'=d''=64$, expansion
            $t=6$, kernel $k=3$. Compute the wide-middle channel count and the block's multiply-adds, then compare
            to one ordinary $3\times3$ convolution doing $64\to64$ on the same map.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Wide channels: $t\,d' = 6\cdot64 = 384$. — _The expand $1\times1$ multiplies the input channels by $t$ (the paper's own 64&rarr;384 example, &sect;3.2)._
- Block cost: $h\,w\,d'\,t(d'+k^2+d'') = 7\cdot7\cdot64\cdot6\,(64+9+64) = 18816\cdot137$. — _Plug into the closed-form block cost; the bracket is expand $d'$ + depthwise $k^2$ + project $d''$._
- Standard conv: $h\,w\,d'\,d''\,k^2 = 49\cdot64\cdot64\cdot9$. — _One full $3\times3$ conv mixes all channels at every position._

**Answer:** The wide middle is $384$ channels. The block costs $18816\cdot137 = \mathbf{2{,}577{,}792}$
                 multiply-adds. A single standard $3\times3$ conv ($64\to64$) costs $49\cdot64\cdot64\cdot9 =
                 \mathbf{1{,}806{,}336}$. So this one block is actually a bit more expensive than one full
                 conv at these widths &mdash; the expansion is not free per block. The win is at the network scale:
                 the residual lets you go deeper and the linear bottleneck lets the data stay thin between blocks,
                 so the whole net beats V1 on accuracy-per-FLOP (Table 4), even though an individual expanded
                 block is not the cheapest thing in isolation.

</details>

**Problem 3.** Why does the paper insist the bottleneck be linear but keep ReLU6 on the expand and depthwise
            layers? Give the information argument and say where the non-linearity is "safe."

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall ReLU zeroes all negatives &mdash; it is a hard, lossy fold of the space. — _Information sent to the negative side of any channel is gone after a ReLU._
- Note the bottleneck is THIN (low-dimensional); the expand/depthwise tensors are WIDE. — _In a thin layer the data manifold fills most of the few available dimensions, so a fold collapses it; in a wide layer the manifold has room and the fold can be 'undone' by other channels._
- Apply the paper's Lemma-1 intuition (&sect;3.1): ReLU preserves a manifold only if it lies in a low-dimensional subspace of a high-dimensional space. — _That condition holds in the wide middle, not on the thin bottleneck._

**Answer:** A ReLU zeroes negatives, which can permanently destroy part of the low-dimensional manifold of
                 interest when the layer is thin &mdash; there are too few channels to route the lost
                 information around the fold. So the thin bottleneck projection is left linear (BatchNorm,
                 no activation). The non-linearity is "safe" only in the wide expand and depthwise layers,
                 where the manifold sits in a high-dimensional space with room to spare, so ReLU6 can add useful
                 non-linearity without collapsing the data (&sect;3.1). This is the &sect;3.2 design rule:
                 non-linearities on the wide layers, linear on the thin ones &mdash; and the ablation in this lesson
                 shows breaking it costs accuracy.

</details>